# 00 — Statik kalibrasyonlu INT8 PTQ (BERT-Tiny / SST-2)

**Proje:** Uç Cihazlar İçin FPGA Tabanlı BERT-Tiny Hızlandırıcısı (BIL 401)

Bu not defteri, FP32 baz modelden başlayıp **eğitim sonrası niceleme (PTQ)** yapar. Model yeniden eğitilmez:
ağırlıklar olduğu gibi kalır, yalnızca kalibrasyon cümleleriyle ölçekler belirlenir ve INT8'e çevrilir.

**Neden bu yöntem (danışman geri bildirimi):**
- *"Önce PTQ, gerekirse QAT"* → bu not defteri birinci adımdır. QAT için `01_qat_colab.ipynb`.
- *"PyTorch'un dinamik nicelemesi donanıma uymaz, kalibrasyonlu statik niceleme kullan"* → `nicele_ptq.py`
  ölçekleri kalibrasyonda **bir kez** belirler ve sabitler; çıkarım sırasında ölçek hesaplanmaz.
- *"FP32–INT8 tahmin uyuşma oranını da raporla"* → her koşu uyuşma oranını ve ayrışan cümleleri yazar.

**Niceleme şeması:** ağırlık = kanal başına simetrik INT8, aktivasyon = tensör başına sabit ölçek
(simetrik ya da asimetrik), bias = INT32 (ölçeği aktivasyon×ağırlık, donanımdaki birikeçle aynı).
Softmax, GELU, LayerNorm, artık toplamlar, QK^T, Softmax·V ve gömme tabloları bu adımda FP32 kalır (G4'ün konusu).

> **Çalışma zamanı:** GPU gerekmez, CPU yeterli ve daha hızlıdır (nicelenmiş çarpım float64 üzerinde yapılıyor,
> T4'te float64 yavaştır). Koşu başına birkaç dakika sürer.

## 0. Ayarlar

In [ ]:
# --- Yollar (Drive) ---
REPO_URL   = "https://github.com/rumeysakolip/bert-tiny-fpga.git"
DRIVE_ROOT = "/content/drive/MyDrive/bitirmeColab"   # Drive'daki çalışma klasörün
FP32_DIR   = f"{DRIVE_ROOT}/fp32_baz"   # içinde model/ ve dogrulama_tahminleri.csv olmalı
OUT_ROOT   = f"{DRIVE_ROOT}/int8_ptq"   # koşu klasörleri buraya yazılır

# --- Veri kaynağı ---
DATA_DIR = None   # None -> GLUE SST-2 Hugging Face'ten (Colab'da DOĞRUSU BU: gerçek 67.349 cümlelik eğitim kümesi)
                  # "/content/drive/MyDrive/.../veri" -> yerel train.tsv/dev.tsv (internet kısıtlıysa)

# --- Koşulacak ayarlar ---
KOSULAR = [
    {"observer": "minmax",     "n_kalib": 128},
    {"observer": "minmax",     "n_kalib": 512},
    {"observer": "percentile", "pct": 99.99, "n_kalib": 128},
    {"observer": "percentile", "pct": 99.99, "n_kalib": 512},
    {"observer": "minmax",     "n_kalib": 512, "asimetrik": True},
    {"observer": "percentile", "pct": 99.99, "n_kalib": 512, "asimetrik": True},
    # Ek deney (sınıflandırıcı ve pooler FP32 kalsın):
    # {"observer": "minmax", "n_kalib": 512, "fp32_katmanlar": "classifier|pooler"},
]

DEVICE         = "cpu"   # "cuda" da çalışır ama float64 yüzünden daha yavaştır; sonuç aynıdır
AGIRLIK_KAYDET = True    # her koşuda int8_agirliklar.npz de yaz (INT8 ağırlıkların elle tutulur hâli)
SEED           = 42      # kalibrasyon cümlelerinin seçim tohumu

## 1. Drive'ı bağla ve girdileri doğrula

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(OUT_ROOT, exist_ok=True)
assert os.path.isdir(f"{FP32_DIR}/model"), (
    f"FP32 model yok: {FP32_DIR}/model\n"
    "Bilgisayarındaki 00-FP32_Egitim/02_Model içeriğini bu klasöre yükle.")
assert os.path.exists(f"{FP32_DIR}/dogrulama_tahminleri.csv"), (
    "dogrulama_tahminleri.csv yok — 00-FP32_Egitim/03_Sonuclar içinden yükle. "
    "Bu dosya FP32 tahminleri; uyuşma oranı bununla hesaplanıyor.")
print("Girdiler hazır:", FP32_DIR)
print("Çıktılar:", OUT_ROOT)

## 2. Kodu indir ve ortamı kur

Depoyu klonlar; Drive'da `bert-tiny-fpga/yazilim/*.py` varsa onların üzerine yazar
(henüz GitHub'a gönderilmemiş değişiklikler için).

In [ ]:
%cd /content
import os, glob, shutil
CALISMA = "/content/calisma"
os.makedirs(f"{CALISMA}/yazilim", exist_ok=True)

GEREKLI = ("nicele_ptq.py", "nicemleme_olcut.py", "egit_fp32.py", "sst2_veri.py")

# 1) Once Drive'daki yazilim/ klasoru (en guncel kopya orada)
n = 0
for p in glob.glob(f"{DRIVE_ROOT}/yazilim/*.py"):
    shutil.copy(p, f"{CALISMA}/yazilim/"); n += 1
print("Drive'dan kopyalanan betik:", n)

# 2) Eksik kalan olursa GitHub deposundan tamamla
if any(not os.path.exists(f"{CALISMA}/yazilim/{g}") for g in GEREKLI):
    if not os.path.exists("/content/bert-tiny-fpga"):
        !git clone {REPO_URL} /content/bert-tiny-fpga
    for p in glob.glob("/content/bert-tiny-fpga/yazilim/*.py"):
        hedef = f"{CALISMA}/yazilim/{os.path.basename(p)}"
        if not os.path.exists(hedef):
            shutil.copy(p, hedef); print("depodan tamamlandi:", os.path.basename(p))

eksik = [g for g in GEREKLI if not os.path.exists(f"{CALISMA}/yazilim/{g}")]
assert not eksik, ("Eksik betik(ler): " + ", ".join(eksik) +
    f"\n-> Bilgisayarindaki 01-INT8_Nicemleme/01_Kod/ icindeki .py dosyalarini "
    f"Drive'da {DRIVE_ROOT}/yazilim/ klasorune yukle.")

os.chdir(CALISMA)
!pip install -q transformers datasets
import torch, transformers
print("Hazir. PyTorch", torch.__version__, "| Transformers", transformers.__version__)
print("Betikler:", sorted(os.listdir(f"{CALISMA}/yazilim")))

## 3. PTQ koşuları

Her koşu sırayla: FP32 referansını yeniden hesaplar → kalibrasyon cümleleriyle ölçekleri belirler →
INT8'e çevirip 872 cümlede değerlendirir → `sonuclar.json`, `olcekler.json`, `int8_tahminleri.csv` yazar.

In [ ]:
import subprocess, time

def ad_uret(k):
    obs = "minmax" if k["observer"] == "minmax" else f"percentile{k.get('pct', 99.99)}"
    return f"{obs}_{'asim' if k.get('asimetrik') else 'sim'}_n{k['n_kalib']}" + ("_fp32cls" if k.get("fp32_katmanlar") else "")

kosular = []
for k in KOSULAR:
    ad = ad_uret(k)
    cmd = ["python", "-u", "yazilim/nicele_ptq.py",
           "--model_dir", f"{FP32_DIR}/model",
           "--fp32_csv",  f"{FP32_DIR}/dogrulama_tahminleri.csv",
           "--out_dir",   OUT_ROOT, "--ad", ad,
           "--observer",  k["observer"], "--n_kalib", str(k["n_kalib"]),
           "--seed", str(SEED), "--device", DEVICE]
    if k["observer"] == "percentile": cmd += ["--pct", str(k.get("pct", 99.99))]
    if k.get("asimetrik"):            cmd.append("--asimetrik")
    if k.get("fp32_katmanlar"):       cmd += ["--fp32_katmanlar", k["fp32_katmanlar"]]
    if DATA_DIR:                      cmd += ["--data_dir", DATA_DIR]
    if AGIRLIK_KAYDET:                cmd.append("--agirlik_kaydet")

    print("\n" + "=" * 72); print("KOŞU:", ad); print("=" * 72)
    t0 = time.time()
    subprocess.run(cmd, check=True)
    print(f"-> {time.time() - t0:.1f} sn")
    kosular.append(f"{OUT_ROOT}/{ad}")

print("\nBiten koşu sayısı:", len(kosular))

## 4. Sonuç tablosu

In [ ]:
import json
from IPython.display import Markdown, display

satirlar = [(os.path.basename(r), json.load(open(f"{r}/sonuclar.json", encoding="utf-8"))["sonuc"]) for r in kosular]
b = satirlar[0][1]

print(f"FP32 baz: {b['dogru_fp32']}/{b['n']}  (%{100 * b['dogruluk_fp32']:.2f})")
print(f"Eşikler  → hedef ≥{b['esikler']['min_dogru_hedef']} | üst sınır ≥{b['esikler']['min_dogru_sinir']} | MLPerf ≥{b['esikler']['min_dogru_mlperf']}")
print(f"1 puan ≈ {b['n'] / 100:.2f} cümle — 1-2 cümlelik farklar gürültüdür.\n")

md_satir = ["| Koşu | INT8 doğru | Kayıp (puan / cümle) | Uyuşma | FP32D→INT8Y | FP32Y→INT8D | Hedef | MLPerf |",
            "|---|---|---|---|---|---|---|---|"]
for ad, s in satirlar:
    md_satir.append(
        f"| {ad} | {s['dogru_int8']} | {s['kayip_puan']} / {s['kayip_cumle']} | %{100 * s['uyusma_orani']:.2f} | "
        f"{s['fp32_dogru_int8_yanlis']} | {s['fp32_yanlis_int8_dogru']} | "
        f"{'evet' if s['hedef_saglandi'] else 'HAYIR'} | {'evet' if s['mlperf_saglandi'] else 'HAYIR'} |")
display(Markdown("\n".join(md_satir)))

en_iyi = max(satirlar, key=lambda x: (x[1]["dogru_int8"], x[1]["uyusma_orani"]))
print("En iyi koşu:", en_iyi[0], "|", en_iyi[1]["karar"])

# Rapor için özet tablo (CSV)
import csv as _csv
with open(f"{OUT_ROOT}/ozet_tablo.csv", "w", newline="", encoding="utf-8") as f:
    w = _csv.writer(f)
    w.writerow(["kosu", "dogru_int8", "kayip_puan", "kayip_cumle", "uyusma_orani",
                "fp32D_int8Y", "fp32Y_int8D", "hedef", "mlperf"])
    for ad, s in satirlar:
        w.writerow([ad, s["dogru_int8"], s["kayip_puan"], s["kayip_cumle"], s["uyusma_orani"],
                    s["fp32_dogru_int8_yanlis"], s["fp32_yanlis_int8_dogru"],
                    s["hedef_saglandi"], s["mlperf_saglandi"]])
print("Özet yazıldı:", f"{OUT_ROOT}/ozet_tablo.csv")

## 5. Karar kuralı — sırada ne var?

Yukarıdaki tablodaki **en kötü** koşuya değil, seçtiğin koşuya bak:

| Kayıp | Ne anlama gelir | Sonraki adım |
|---|---|---|
| **≤ 1 puan** | Hedef sağlandı | QAT'a gerek yok. G4'e geç (Softmax/GELU/LayerNorm bit genişliği). |
| **1 – 2 puan** | Sınır içinde ama hedef aşıldı | Önce hata analizi: hangi katman bozuyor, aktivasyon aykırı değerleri nerede? Sonra QAT. |
| **> 2 puan** | Sınır aşıldı | QAT (`01_qat_colab.ipynb`) ve katman bazlı hassasiyet analizi. |

Not: Birden çok ayar deneyip en iyisini doğrulama kümesine bakarak seçmek iyimser bir seçimdir;
raporda kaç ayar denediğini ve seçim gerekçeni yaz. Koşular arası fark 1–2 cümleyse
"bu ayar daha iyi" deme — gürültüdür.

## 6. Drive'a yazımı tamamla

In [ ]:
drive.flush_and_unmount()
print("Drive senkronlandı:", OUT_ROOT)